---
![LOGO UTN FRCU](https://frcu.utn.edu.ar/images/recursos/logos/MARCA_FACULTAD_REGIONAL_NEGRO.png)
# BiblioIA: Sistema de Gestión de Biblioteca con Agente de Inteligencia Artificial"
- Materia: Base de Datos
- Año / Carrera: 3° año — Ing. en Sistemas de Información
- Docentes: Ing. Francisco Fazzio, Ing. Pablo Colombo
- Ciclo lectivo: 2026
- Grupo : Grupo Pandas
- Integrantes : Cheveste Baloni Ulises, Delfino Jeremias, Gallegos Marco, Ocampo Emmanuel


---
## 1. Configuración
carga .env, conecta a MySQL.

### 1.1 Instalación de librerías

In [ ]:
pip install mysql-connector-python pandas google-genai python-dotenv

### 1.2 Configuración

In [2]:
import os
from dotenv import load_dotenv
import mysql.connector
import pandas as pd
import ipywidgets as widgets
from google import genai

load_dotenv()
DB_CONF = {
    "host": os.getenv("DB_HOST", "localhost"),
    "port": os.getenv("DB_PORT",  "3306"),
    "database": os.getenv("DB_NAME", "biblioia"),
    "user": os.getenv("DB_USER", "root"),
    "password": os.getenv("DB_PASSWORD", "")
}

LLM_API_KEY= os.getenv("LLM_API_KEY", "")




---
## 2. Inspección del esquema
Carga structure de tablas para el prompt

In [7]:
sys_prompt = """ Sos un Administrador de bases de datos de una biblioteca. Responde las consultas en lenguaje natural únicamente devolviendo una consulta en MySQL
    sabiendo que el schema de la base de datos en MySQL de esta biblioteca es el siguiente:
    1. **Genero** (Categorías de libros)
    - id_genero (PK, auto_increment)
    - nombre (VARCHAR 100, UNIQUE, NOT NULL)
    - descripcion (TEXT)

    2. **Autor** (Autores de libros)
    - id_autor (PK, auto_increment)
    - nombre (VARCHAR 100, NOT NULL)
    - apellido (VARCHAR 100, NOT NULL)
    - nacionalidad (VARCHAR 100)

    3. **Socio** (Miembros de la biblioteca)
    - id_socio (PK, auto_increment)
    - dni (VARCHAR 20, UNIQUE, NOT NULL)
    - nombre (VARCHAR 100, NOT NULL)
    - apellido (VARCHAR 100, NOT NULL)
    - email (VARCHAR 150, UNIQUE, NOT NULL)
    - fecha_alta (DATE, NOT NULL)
    - estado (VARCHAR 20, DEFAULT 'activo') - Valores: 'activo', 'suspendido', 'baja'

    4. **Libro** (Catálogo de libros)
    - isbn (VARCHAR 20, PK)
    - titulo (VARCHAR 255, NOT NULL)
    - anio_publicacion (INT)
    - stock_total (INT, DEFAULT 0)
    - stock_disponible (INT, DEFAULT 0)
    - id_genero (FK → Genero)
    
    5. **Libro_genero** (Relación muchos-a-muchos entre Libros y Generos)
    - id_genero (FK → Genero)
    - isbn (FK → Libro)
    - PK: (id_genero, isbn)
    
    6. **Autor_libro** (Relación muchos-a-muchos entre Autores y Libros)
    - id_autor (FK → Autor)
    - isbn (FK → Libro)
    - PK: (id_autor, isbn)

    7. **Ejemplar** (Copias físicas de libros)
    - id_ejemplar (PK, auto_increment)
    - isbn (FK → Libro, NOT NULL)
    - nro_ejemplar (INT, NOT NULL)
    - estado_fisico (VARCHAR 20, DEFAULT 'bueno') - Valores: 'bueno', 'regular', 'reparacion', 'baja'

    8. **Prestamo** (Registro de préstamos)
    - id_prestamo (PK, auto_increment)
    - id_socio (FK → Socio, NOT NULL)
    - id_ejemplar (FK → Ejemplar, NOT NULL)
    - fecha_prestamo (DATE, NOT NULL)
    - fecha_vencimiento (DATE, NOT NULL)
    - fecha_devolucion (DATE, NULL)
    - estado (VARCHAR 20, DEFAULT 'activo') - Valores: 'activo', 'devuelto', 'vencido'

    9. **Sancion** (Sanciones a socios)
    - id_sancion (PK, auto_increment)
    - id_socio (FK → Socio, NOT NULL)
    - tipo (VARCHAR 50, NOT NULL)
    - fecha_inicio (DATE, NOT NULL)
    - fecha_fin (DATE, NOT NULL)
    - motivo (TEXT)

    INSTRUCCIONES IMPORTANTES:
    - Cuando generes consultas SQL, asegúrate de que sean válidas para MySQL.
    - Usa INNER JOIN para relaciones requeridas y LEFT JOIN cuando sea necesario.
    - Siempre valida que los tipos de datos sean correctos.
    - Tus respuestas deben contener únicamente la consulta SQL, sin ningún tipo de texto adicional, pues serán inmediatamente cargadas a la base de datos y no pueden contener símbolos que lo impidan. 

    EJEMPLOS DE CONSULTAS EN LENGUAJE NATURAL Y RESPUESTAS EN SQL:
    Para la consulta:
    Muestrame todos los libros de Franz Kafka que se hayan publicado luego del año 1950.

    Una respuesta en SQL adecuada sería:
    SELECT l.*
    FROM Libro l
    INNER JOIN Autor_libro al ON l.isbn = al.isbn
    INNER JOIN Autor a ON al.id_autor = a.id_autor
    WHERE a.nombre = 'Franz' 
    AND a.apellido = 'Kafka' 
    AND l.anio_publicacion > 1950;

    Para la consulta:
    Muestrame todos los autores que hayan publicado por lo menos 5 libros del género ciencia ficción
    
    Una respuesta en SQL adecuada sería:
    SELECT a.id_autor, a.nombre, a.apellido
    FROM Autor a
    INNER JOIN Autor_libro al ON a.id_autor = al.id_autor
    INNER JOIN Libro_genero lg ON al.isbn = lg.isbn
    INNER JOIN Genero g ON lg.id_genero = g.id_genero
    WHERE g.nombre = 'Ciencia Ficción'
    GROUP BY a.id_autor, a.nombre, a.apellido
    HAVING COUNT(DISTINCT al.isbn) >= 5;

    """

---
## 3. Función text_to_sql
Llama a la API y retorna SQL.

In [ ]:
# Cargar prompt y toda la cuestión lol
def conectar_DB(query):
    try:
        db= mysql.connector.connect(**DB_CONF)
        resultado = pd.read_sql(query,db)
        db.close()
        return resultado
    except Exception as e:
        db.close()
        print(str(e))

def consultar(consulta_natural):
    
    client = genai.Client(api_key=LLM_API_KEY)

    response = client.models.generate_content(
        model="gemini-3.5-flash",
        contents=f"{sys_prompt}\n\nConsulta: {consulta_natural}"
    )
    client.close()
    tabla = conectar_DB(response.text)
    return tabla

def recomendar(consulta_natural):
    
    client = genai.Client(api_key=LLM_API_KEY)

    response = client.models.generate_content(
        model="gemini-3.5-flash",
        contents=f"Consulta: {consulta_natural}"
    )
    client.close()
    return response.text


In [ ]:
def DB_cargar_lista(tabla, columna):
    try:
        db=  mysql.connector.connect(**DB_CONF)
        query = "select "+ columna +" from "+tabla
        cursor = db.cursor()
        cursor.execute(query)
        resultado = list()
        for row in cursor:
            resultado.append(row)
        db.close() 
        return resultado
    except Exception as e:
        db.close()
        print(str(e))

---
## 4. Función agente_responder(pregunta)
Orquesta todo y muestra resultado.

In [6]:
# consulta_natural = input('Ingrese su consulta: ')
# consultar(consulta_natural)

---
## 5. Demo interactivo
Widget ipywidgets o input() para preguntas libres

### 5.1. Buscador de libros según autor

In [ ]:
autores = DB_cargar_lista('Autor','nombre, apellido')
l = list()
for row in autores:
    l.append(row[0]+' '+row[1])
    #print(row)
d = widgets.Dropdown(
    options = l,
    description = 'Autor: '
)
b = widgets.Button(
    description='Buscar libros'
)
w = widgets.HBox([d,b])
out = widgets.Output()
out2 = widgets.Output()
out2.append_stdout("Buscador de libros según autor")
display(out2,w,out)
def on_button_clicked(b):
    with out:
        out.clear_output()
        tabla = consultar('Lista libros de '+d.value+' y la cantidad de ejemplares en stock de cada uno')
        display(tabla)
b.on_click(on_button_clicked)

### 5.2 Barra de búsqueda de libros 

In [ ]:
t = widgets.Textarea(
    description = "Consulta: ",
    placeholder = "Escriba su consulta",
    value = "Muestrame todos los libros que se hayan publicado luego del año 1980" 
)
b = widgets.Button(
    description = "Consultar"
)
o = widgets.Output()
hbox = widgets.HBox([t,b])
display(hbox,o)
def on_button_clicked(b):

    with o:
        o.clear_output()
        tabla = consultar(t.value)
        display(tabla)
b.on_click(on_button_clicked)

---
## 6. Módulo de recomendaciones: función recomendar_para(id_socio)

In [12]:
id_socio = input("Ingrese su ID de socio: ")
query_leidos = "call LibrosLeidos("+id_socio+")"
query_similares = "call Recomendar("+id_socio+")"
tabla_leidos = conectar_DB(query_leidos)
tabla_leidos.to_csv("libros_Leidos.csv")
tabla_similares = conectar_DB(query_similares)
tabla_similares.to_csv("libros_similares.csv")
leidos = tabla_leidos.to_json()
similares = tabla_similares.to_json()
prompt_recomendacion = """ En base a los libros leídos de la siguiente tabla json: 
"""+leidos+"""
, haz un listado que incluya cada libro la siguiente tabla (similares) json:
"""+similares+"""
, y justifica por qué lo recomendarías,
teniendo en cuenta que los libros de la tabla de similares comparten género o autores con los libros de la tabla de leídos.
El formato de la lista debe ser del estilo:
- Títlo del libro: justificación de la recomendación
"""
print("Libros Leídos: ")
display(tabla_leidos)
print("Libros similares: ")
display(tabla_similares)
print("Recomendaciones: ")

print(recomendar(prompt_recomendacion))

Libros Leídos: 


/tmp/ipykernel_4150/616317618.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  resultado = pd.read_sql(query,db)
/tmp/ipykernel_4150/616317618.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  resultado = pd.read_sql(query,db)


,titulo,generos,autor
0,Fundación,Literatura Clásica,Franz Kafka
1,Muerte en el Nilo,Historia y Cultura Argentina,Jorge Luis Borges


Libros similares: 


,isbn,titulo,stock_disponible
0,978-03,2001: Una odisea espacial,2
1,978-09,Asesinato en el Orient Express,3
2,978-04,Cita con Rama,2
3,978-17,El Nombre de la Rosa,3
4,978-18,El Péndulo de Foucault,2
5,978-13,El Resplandor,3
6,978-12,El Sabueso de los Baskerville,2
7,978-11,Estudio en Escarlata,2
8,978-15,La Llamada de Cthulhu,2
9,978-02,"Yo, Robot",2


Recomendaciones: 
Para realizar las recomendaciones, es importante señalar que la tabla de libros leídos presenta una mezcla en sus metadatos (por ejemplo, asocia *Fundación* a Franz Kafka y "Literatura Clásica", y *Muerte en el Nilo* a Jorge Luis Borges e "Historia y Cultura Argentina"). 

Tomando en cuenta tanto los datos proporcionados en tu JSON como los **autores y géneros reales** de estas obras (ya que *Fundación* es de ciencia ficción por Isaac Asimov y *Muerte en el Nilo* es una novela policial de Agatha Christie), aquí tienes el listado de recomendaciones de la tabla de similares:

- **Yo, Robot**: Se recomienda porque comparte autor real (Isaac Asimov) y género (ciencia ficción) con *Fundación*. Además, al igual que los autores mencionados en tu tabla (como Kafka o Borges), es una obra fundamental de la literatura universal.
- **Asesinato en el Orient Express**: Se recomienda porque comparte autora real (Agatha Christie) y género (misterio/policial) con *Muerte en el Nilo*, 